In [ ]:
!pip install pygrog[dev]


# Interoperability with mrpro, sigpy and deepinverse

PyGROG ships three thin adapter layers so that
:class:`~pygrog.operator.SparseFFT` (and any gadget stacked on top of it)
can be used directly inside third-party reconstruction frameworks:

* **mrpro** — :class:`~pygrog.interop.GrogLinearOp` wraps SparseFFT as an
  ``mrpro.operators.LinearOperator`` with explicit autograd support.
* **sigpy** — :class:`~pygrog.interop.GrogLinop` wraps SparseFFT as a
  ``sigpy.linop.Linop`` with a working ``.H`` adjoint property.
* **deepinverse** — :class:`~pygrog.interop.GrogLinearPhysics` subclasses
  ``deepinv.physics.LinearPhysics``, enabling all deepinv algorithms
  (unrolled networks, PnP, RED, …) to be used directly.

For differentiable use in plain PyTorch (without deepinv) the lower-level
:func:`~pygrog.interop.grog_measure` and
:func:`~pygrog.interop.grog_backproject` autograd functions are also
available.

Each adapter is imported lazily so that missing optional dependencies raise
an informative error only when the adapter is actually instantiated.

All data are **synthetic** — no scanner files are required.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from brainweb_dl import get_mri
from mrinufft import get_operator, initialize_2D_spiral
from mrinufft.density import voronoi
from pygrog.calib import GrogInterpolator
from pygrog.operator import SparseFFT

## Shared setup: BrainWeb phantom, spiral trajectory, GROG plan

k-space is always simulated via mri-nufft; PyGROG's GROG interpolation then
maps it onto a sparse Cartesian grid.  Pre-multiplying by
``grog.plan.pre_weights`` once before the solver loop ensures
:class:`~pygrog.operator.SparseFFT` ``.forward`` / ``.adjoint`` satisfy the
adjointness condition throughout.



In [ ]:
image_np = get_mri(0, "T1")
image_np = np.flip(image_np, axis=(0, 2))[90].astype(np.float32)
image_np /= image_np.max() + 1e-8
image_shape = image_np.shape
n_coils = 16


def _synthetic_smaps(shape, n_coils=4):
    ny, nx = shape
    yy, xx = np.mgrid[-1 : 1 : ny * 1j, -1 : 1 : nx * 1j]
    smaps = []
    for angle in np.linspace(0.0, 2.0 * np.pi, n_coils, endpoint=False):
        cx, cy = np.cos(angle), np.sin(angle)
        gauss = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * 0.45**2))
        phase = np.exp(1j * (cx * xx + cy * yy))
        smaps.append(gauss * phase)
    smaps = np.asarray(smaps, dtype=np.complex64)
    smaps /= np.sqrt((np.abs(smaps) ** 2).sum(0, keepdims=True)) + 1e-12
    return smaps


smaps = _synthetic_smaps(image_shape, n_coils=n_coils)

samples = initialize_2D_spiral(Nc=48, Ns=600, nb_revolutions=10).astype(
    np.float32
)
density = voronoi(samples)

nufft = get_operator("finufft")(
    samples=samples,
    shape=image_shape,
    n_coils=n_coils,
    smaps=smaps,
    density=density,
    squeeze_dims=True,
)
# k-space simulated entirely via mri-nufft (never via GrogInterpolator)
kspace_nufft = nufft.op(image_np.astype(np.complex64))  # (n_coils, n_samples)

# GROG calibration: use 24x24 k-space centre of coil images
coords = (samples * np.asarray(image_shape, dtype=np.float32)).astype(np.float32)
coil_calib = smaps * image_np[None, ...]
calib_cart_full = np.fft.fftshift(
    np.fft.fftn(np.fft.ifftshift(coil_calib, axes=(-2, -1)), axes=(-2, -1)),
    axes=(-2, -1),
).astype(np.complex64)
calib_size = 24
cy, cx = image_shape[0] // 2, image_shape[1] // 2
calib_cart = calib_cart_full[
    :,
    cy - calib_size // 2 : cy + calib_size // 2,
    cx - calib_size // 2 : cx + calib_size // 2,
]
grog = GrogInterpolator(
    shape=image_shape, coords=coords, kernel_width=2, oversamp=1.25, image_shape=image_shape
)
grog.calc_interp_table(calib_cart, lamda=0.01, precision=1)

# Interpolate to sparse Cartesian and pre-weight (once, before any iterative loop)
kspace_nc_shaped = kspace_nufft.astype(np.complex64).reshape(n_coils, *samples.shape[:2])
kspace_sparse_raw = grog.interpolate(kspace_nc_shaped, ret_image=False)
kspace_sparse = torch.as_tensor(np.asarray(kspace_sparse_raw))
kspace_sparse = kspace_sparse * grog.plan.pre_weights.to(kspace_sparse.dtype).unsqueeze(0)

# SparseFFT operator wrapping the GROG plan (with smaps for coil combination)
op = SparseFFT(plan=grog.plan, smaps=torch.as_tensor(smaps))

print(f"image shape   : {image_shape}")
print(f"sparse k-space: {kspace_sparse.shape}  (pre-weighted)")
print(f"grid shape    : {op.grid_shape}")

## deepinverse adapter — GrogLinearPhysics

:class:`~pygrog.interop.GrogLinearPhysics` subclasses
``deepinv.physics.LinearPhysics`` so that SparseFFT slots into any deepinv
reconstruction algorithm (unrolled networks, PnP, RED, …).  Gradients are
provided by explicit :class:`torch.autograd.Function` subclasses — not by
automatic differentiation through the GROG kernels.

<div class="alert alert-info"><h4>Note</h4><p>``deepinv`` must be installed separately (``pip install deepinv``).
   When not available the lazy class factory raises an informative
   ``ImportError``.</p></div>



In [ ]:
from pygrog.interop import GrogLinearPhysics

try:
    physics = GrogLinearPhysics(op)

    # Forward (adjoint NUFFT): pre-weighted sparse k-space -> image
    img_deepinv = physics.A_adjoint(kspace_sparse)
    print(f"\nGrogLinearPhysics.A_adjoint output: {img_deepinv.shape}")

    # Adjoint (forward NUFFT): image -> sparse k-space
    ksp_deepinv = physics.A(img_deepinv)
    print(f"GrogLinearPhysics.A   output shape: {ksp_deepinv.shape}")

    # Gradient flows through A_adjoint
    ksp_grad = kspace_sparse.clone().requires_grad_(True)
    img_out = physics.A_adjoint(ksp_grad)
    img_out.abs().sum().backward()
    print(f"Gradient populated on ksp_grad     : {ksp_grad.grad is not None}")

except ImportError as exc:
    print(f"\ndeepinv not available — skipping GrogLinearPhysics demo: {exc}")

### Plain-PyTorch autograd functions

For use without deepinv, :func:`~pygrog.interop.grog_measure` (image → k-space)
and :func:`~pygrog.interop.grog_backproject` (k-space → image) are the
underlying autograd functions.



In [ ]:
from pygrog.interop import grog_measure

# grog_measure operates on a plain image without coil combination — use a
# no-smaps operator so that gradient shapes are consistent.
op_no_smaps = SparseFFT(plan=grog.plan)
image_t = torch.as_tensor(image_np[None].astype(np.complex64))  # (1, *image_shape)
img_grad = image_t.clone().requires_grad_(True)
ksp_out = grog_measure(img_grad, op_no_smaps)
ksp_out.abs().sum().backward()
print(f"\ngrog_measure output shape : {ksp_out.shape}")
print(f"Gradient on img_grad      : {img_grad.grad is not None}")

## sigpy adapter — GrogLinop

:class:`~pygrog.interop.GrogLinop` returns a ``sigpy.linop.Linop`` with a
working ``.H`` property, so it can be composed with other sigpy operators
using ``*``, ``+``, and scalar multiplication.

<div class="alert alert-info"><h4>Note</h4><p>``sigpy`` must be installed separately (``pip install sigpy``).  When
   sigpy is not available this block prints an informative message and
   continues.</p></div>



In [ ]:
from pygrog.interop import GrogLinop

try:
    linop = GrogLinop(op)
    print(f"\nGrogLinop    ishape={linop.ishape}, oshape={linop.oshape}")
    print(f"GrogLinop.H  ishape={linop.H.ishape}, oshape={linop.H.oshape}")

    # Forward (adjoint NUFFT): pre-weighted sparse k-space -> image
    ksp_np_sigpy = kspace_sparse.numpy()
    img_sigpy = linop * ksp_np_sigpy
    print(f"linop output shape   : {img_sigpy.shape}")

    # Adjoint (forward NUFFT): image -> sparse k-space
    img_np_sigpy = np.abs(img_sigpy)
    ksp_sigpy = linop.H * img_np_sigpy.astype(np.complex64)
    print(f"linop.H output shape : {ksp_sigpy.shape}")

except ImportError as exc:
    print(f"\nsigpy not available — skipping GrogLinop demo: {exc}")

## mrpro adapter — GrogLinearOp

:class:`~pygrog.interop.GrogLinearOp` returns an
``mrpro.operators.LinearOperator`` with autograd support via explicit
:class:`torch.autograd.Function` wrappers (no ``adjoint_as_backward``).
It participates in mrpro operator algebra (``@``, ``+``, adjoint via ``.H``)
and can be passed directly to mrpro solvers (e.g. ``ConjugateGradient``).



In [ ]:
from pygrog.interop import GrogLinearOp

try:
    mrpro_op = GrogLinearOp(op)
    print(f"\nGrogLinearOp ready: {type(mrpro_op).__mro__[1].__name__}")

    # Forward (adjoint NUFFT): pre-weighted sparse k-space -> image
    (img_mrpro,) = mrpro_op.forward(kspace_sparse)
    print(f"mrpro forward output shape: {img_mrpro.shape}")

    # Adjoint (forward NUFFT): image -> sparse k-space
    (ksp_mrpro,) = mrpro_op.adjoint(img_mrpro)
    print(f"mrpro adjoint output shape: {ksp_mrpro.shape}")

    # Normal operator A^H A
    AHA = mrpro_op.H @ mrpro_op
    (img_aha,) = AHA.forward(kspace_sparse)
    print(f"Normal operator (A^H A) output shape: {img_aha.shape}")

except ImportError as exc:
    print(f"\nmrpro not available — skipping GrogLinearOp demo: {exc}")

## Summary

.. list-table:: Adapter summary
   :header-rows: 1
   :widths: 25 40 35

   * - Adapter
     - Interface
     - Use case
   * - ``GrogLinearPhysics``
     - ``deepinv.physics.LinearPhysics``
     - deepinv algorithms (PnP, …)
   * - ``grog_measure`` / ``grog_backproject``
     - ``torch.autograd.Function``
     - plain PyTorch gradient recon
   * - ``GrogLinop``
     - ``sigpy.linop.Linop``
     - sigpy CG / PDHG solvers
   * - ``GrogLinearOp``
     - ``mrpro.operators.LinearOperator``
     - mrpro CG / PDHG solvers



In [ ]:
plt.show()